In [1]:
# Transformers example -> Sorting a sequence of numbers


In [2]:
import flax.nnx as nnx
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn import PosEncode, Transformer
from probjax.nn.layers.attention import flex_attention
from probjax.nn.pallas_kernels.attention_mask_bias import SeqLenMask

In [3]:
VOCAB_SIZE = 10

In [4]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)

    sequences_sorted = jnp.sort(sequences, axis=-2)

    return sequences, sequences_sorted

inputs, labels = generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

In [5]:
key = jrandom.PRNGKey(0)

In [6]:
class Model(nnx.Module):

    def __init__(self, dim,rngs, dropout_rate=0.):
        self.embed = nnx.Embed(VOCAB_SIZE, dim, rngs=rngs)
        self.pos_embed = PosEncode(dim,rngs=rngs)
        self.transformer = Transformer(dim, 1,4,10, widening_factor=2,rngs=rngs, attention_fn=flex_attention, dropout_rate=dropout_rate)
        self.output = nnx.Linear(dim, VOCAB_SIZE, rngs=rngs)

    def __call__(self, x, deterministic=False, mask=None):
        x = self.embed(x)
        x = jnp.squeeze(x,axis=-2)
        x = self.pos_embed(x)
        x = self.transformer(x,deterministic=deterministic, mask=mask)
        x = self.output(x)
        return x


In [7]:
model = Model(50, rngs=nnx.Rngs(0), dropout_rate=0.0)

In [8]:
nnx.display(model)

/Users/manug/src/probjax/.venv/lib/python3.13/site-packages/treescope/renderers.py:314: UserWarning: Ignoring error while formatting value of type <class 'flax.nnx.helpers.List'> with <function handle_via_treescope_repr_method at 0x11a797f60>:
Traceback (most recent call last):
  File "/Users/manug/src/probjax/.venv/lib/python3.13/site-packages/treescope/renderers.py", line 290, in _render_subtree
    maybe_result = handler(node=node, path=path, subtree_renderer=rec)
  File "/Users/manug/src/probjax/.venv/lib/python3.13/site-packages/treescope/_internal/handlers/custom_type_handlers.py", line 65, in handle_via_treescope_repr_method
    return treescope_repr_method(path, subtree_renderer)
  File "/Users/manug/src/probjax/.venv/lib/python3.13/site-packages/flax/nnx/pytreelib.py", line 702, in __treescope_repr__
    if name.startswith('_'):
       ^^^^^^^^^^^^^^^
AttributeError: 'int' object has no attribute 'startswith'

  warnings.warn(


In [9]:
params = nnx.state(model, nnx.Param)

In [10]:
optimizer = optax.adam(5e-4)
opt_state = optimizer.init(params)

In [17]:

from probjax.nn.pallas_kernels.utils import materialize_mask


max_train_seq = 64
def loss_fn(params, key):
    nnx.update(model, params)
    key, key_sub = jax.random.split(key)
    inp_data, labels = generate_data(key_sub, 128, max_train_seq, vocab_size=VOCAB_SIZE)
    lengths = jax.random.randint(key, (128,), 1, max_train_seq)
    mask = SeqLenMask(lengths)
    logits = model(inp_data, mask=mask)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    loss = optax.softmax_cross_entropy(logits, labels)
    length_mask = jnp.arange(max_train_seq)[None, :] < lengths[:, None]
    loss = (loss * length_mask).sum() / length_mask.sum()
    return loss

@jax.jit
def acc(params, inputs, outputs):
    nnx.update(model, params)
    inp_data, labels = inputs, outputs
    logits = model(inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, key, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [18]:
key = jrandom.PRNGKey(0)

In [16]:

for i in range(10_000):
    key, subkey, key2 = jrandom.split(key, 3)
    loss, params, opt_state = update(params,key, opt_state)
    if (i % 1000) == 0:
        inputs, labels = generate_data(key, 32,32, vocab_size=VOCAB_SIZE)
        accuracy = acc(params, inputs, labels)
        print(accuracy, loss)

0.12695312 0.4157355
0.12109375 0.42797434


KeyboardInterrupt: 

In [19]:
model.eval()
nnx.update(model, params)

In [21]:
input = jax.random.randint(key+1, (1, 15,1),0, 10,dtype=jnp.int32)
outputs = model(input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])
print(jnp.sort(input[0,...,0]))

[0 7 0 5 7 8 2 7 0 6 3 3 6 4 3]
[0 0 0 0 0 0 0 0 0 0 0 0 1 2 2]
[0 0 0 2 3 3 3 4 5 6 6 7 7 7 8]


In [ ]:
jnp.allclose(outputs.argmax(-1)[0], jnp.sort(input[0,...,0]))

Array(False, dtype=bool)